# Model Evaluation

In [21]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import pickle as pkl
from scipy import sparse
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import roc_curve
from sklearn.metrics import auc
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_curve


from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn import svm
from sklearn.tree import DecisionTreeClassifier


from sklearn.externals import joblib

### Load the X (Features and the target variables (y)

We will Load the X features with the two types:
- The X data for TF_IDF featureres
- The X data for skip gram

Also, we load the target variable y as well and will split the data to make it fit.

In [ ]:
#Load the features and the X TF_IDF and X_ SKIPGRAM
feature_mtx = np.load('model-features/feature_matrix.npy')
x_tfidf_sparse = sparse.load_npz('model-features/tf-idf-matrix.npz')
x_word_embed = np.load('model-features/wordembedding_feature.npy')


#________________________________________________
#the Xs for TFIDF feature
x_tfidf = np.concatenate((feature_mtx, x_tfidf_sparse.toarray()), axis=1)
#Xs for embedding features
x_embedding = np.concatenate((feature_mtx, x_word_embed), axis=1)

y = np.load('model-features/label.npy')
#________________________________________________

#split the X_TFIDF data with the labels (target variable)
x_tf_train, x_tf_test, y_tf_train, y_tf_test = train_test_split(x_tfidf, y, test_size=0.3, random_state=0)
#split the X_skipgram with the labels (target variables)
x_sg_train, x_sg_test, y_sg_train, y_sg_test = train_test_split(x_embedding, y, test_size=0.3, random_state=0)

### Fitting the Logistic Regression Classifiers with the best hyperparameters

We load two Logistic regression classifiers that contain best hyperparameters:
- First Classifier for Skip Gram Features
- Second Classifer for TF_IDF Features

In [ ]:
f = open('Classifier/LR_skipgram_clf.pkl', 'rb')
#LR: Logistic Regression      SG: Skipgram      HP: Hyper Parameters
lr_hp_sg = pkl.load(f)
lr_sg = LogisticRegression(C=lr_hp_sg['C'], penalty=lr_hp_sg['penalty'])
lr_sg = lr_sg.fit(x_sg_train, y_sg_train)

f = open('Classifier/LR_TF-ID_clf.pkl', 'rb')
#Load the file that will give us Logistic classifier
lr_tfidf = pkl.load(f)
lr_tfidf = lr_tfidf.fit(x_tf_train, y_tf_train)

### Fitting the Random Forest with the best hyperparameters

We load two Random Forest classifiers that contain best hyperparameters:
- First Classifier for Skip Gram Features
- Second Classifer for TF_IDF Features

In [ ]:
f = open('Classifier/RF_skipgram_clf.pkl', 'rb')
#RF: Random Forest      SG: Skipgram      HP: Hyper Parameters
rf_sg = pkl.load(f)
rf_sg = rf_sg.fit(x_sg_train, y_sg_train)

f = open('Classifier/RF_TF-ID_clf.pkl', 'rb')
#load the file that will give us Random Forest classifier
rf_tfidf = pkl.load(f)
rf_tfidf = rf_tfidf.fit(x_tf_train, y_tf_train)

### Fitting the SVM with the best hyperparameters

We load two SVM classifiers that contain best hyperparameters:
- First Classifier for Skip Gram Features
- Second Classifer for TF_IDF Features

In [ ]:
f = open('Classifier/SVM_skipgram_clf.pkl', 'rb')
#SVM: Support Vector Machine      SG: Skipgram      HP: Hyper Parameters
svm_sg = pkl.load(f)
svm_sg = svm_sg.fit(x_sg_train, y_sg_train)

f = open('Classifier/SVM_TF-ID_clf.pkl', 'rb')
#load the file that will give us SVM classifier
svm_tfidf = pkl.load(f)
svm_tfidf = svm_tfidf.fit(x_tf_train, y_tf_train)

### Fitting the Decision Tree with the best hyperparameters

We load two Decision Tree classifiers that contain best hyperparameters:
- First Classifier for Skip Gram Features
- Second Classifer for TF_IDF Features

In [ ]:
f = open('Classifier/decision_tree_sg_clf.pkl', 'rb')
#DT: Decision Tree      SG: Skipgram      HP: Hyper Parameters
dt_sg = pkl.load(f)
dt_sg = dt_sg.fit(x_sg_train, y_sg_train)

f = open('Classifier/decision_tree_sg_clf.pkl', 'rb')
#load the file that will give us Decision Tree classifier
dt_tfidf = pkl.load(f)
dt_tfidf = dt_tfidf.fit(x_tf_train, y_tf_train)

### Show the Confusion Matrix

Draw the confusion matrix for all the models that we fitted and compare between them

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(18, 18))
fig.suptitle('Skip Gram                    VS                    TF-IDF', fontsize=14)

def draw_cm(clf, x_test, y_test, i, j, clf_name):
    y_pred = clf.predict(x_test)
    ax = sns.heatmap(confusion_matrix(y_pred, y_test), ax=axes[i, j], annot=True, cmap="Blues", fmt='g')
    ax.xaxis.set_ticklabels(['-1', '1']); ax.yaxis.set_ticklabels(['-1', '1']);
    axes[i, j].set(title=clf_name)
    return ax

lr1 = draw_cm(lr_sg,      x_sg_test, y_sg_test, 0, 0, "Logistic Regression")
lr2 = draw_cm(lr_tfidf,   x_tf_test, y_tf_test, 0, 1, "Logistic Regression")

rf1 = draw_cm(rf_sg,      x_sg_test, y_sg_test, 1, 0, "Random Forest")
rf2 = draw_cm(rf_tfidf,   x_tf_test, y_tf_test, 1, 1, "Random Forest")

svm1 = draw_cm(svm_sg,    x_sg_test, y_sg_test, 2, 0, "SVM")
svm2 = draw_cm(svm_tfidf, x_tf_test, y_tf_test, 2, 1, "SVM")

dt1 = draw_cm(dt_sg,      x_sg_test, y_sg_test, 3, 0, "Decision Tree")
dt2 = draw_cm(dt_tfidf,   x_tf_test, y_tf_test, 3, 1, "Decision Tree")

plt.show()

### Plotting Precision Recall Curve

We plot the Precision Recall Curve to show the effectiveness of the model

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(13, 25))
fig.suptitle('Skip Gram                    VS                    TF-IDF', fontsize=14)

def plot_precision_recall_curve(classifier, x_test, y_test, i, j, clf_name):
    y_score = classifier.predict_proba(x_test)[:, 1]

    precision, recall, thresholds = precision_recall_curve(y_test, y_score)
    axes[i, j].plot(recall, precision, label=clf_name)
    axes[i, j].set(xlabel='Recall', ylabel='Precision')
    axes[i, j].legend(loc='lower left')

    plt.xlim([-0.01, 1.0])
    plt.ylim([0.0, 1.1])
    axes[i, j].plot([1, 0], [0, 1], 'k--')
    plt.title('Precision-Recall Curve')


lr1 = plot_precision_recall_curve(lr_sg,    x_sg_test, y_sg_test, 0, 0, 'Logistic Regression')
lr2 = plot_precision_recall_curve(lr_tfidf, x_tf_test, y_tf_test, 0, 1, 'Logistic Regression')

rf1 = plot_precision_recall_curve(rf_sg,    x_sg_test, y_sg_test, 1, 0, 'Random Forest')
rf2 = plot_precision_recall_curve(rf_tfidf, x_tf_test, y_tf_test, 1, 1, 'Random Forest')

dt1 = plot_precision_recall_curve(dt_sg,    x_sg_test, y_sg_test, 2, 0, 'Decision Tree')
dt2 = plot_precision_recall_curve(dt_tfidf, x_tf_test, y_tf_test, 2, 1, 'Decision Tree')

svm1 = plot_precision_recall_curve(svm_sg,    x_sg_test, y_sg_test, 3, 0, 'SVM')
svm2 = plot_precision_recall_curve(svm_tfidf, x_tf_test, y_tf_test, 3, 1, 'SVM')

plt.show()

### Plotting Receiver Operating Characteristic (ROC)

To show the effeciency of the model and if it is overfitted or not

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(13, 25))
fig.suptitle('Skip Gram                    VS                    TF-IDF', fontsize=14)

def plot_roc(classifier, x_test, y_test, i, j, clf_name):
    y_score = classifier.predict_proba(x_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_score, pos_label=1)
    axes[i, j].plot([0, 1], [0, 1], 'k--')
    roc_auc = auc(fpr, tpr)
    axes[i, j].plot(fpr, tpr, label=str(clf_name) + ' (AUC = %0.3f)' % roc_auc)
    axes[i, j].legend(loc='lower right')
    axes[i, j].set(xlabel='False Positive Rate', ylabel='True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC)')
    plt.xlim([-0.01, 1.0])
    plt.ylim([0.0, 1.0])


lr1 = plot_roc(lr_sg,     x_sg_test, y_sg_test, 0, 0, 'LR SG')
lr2 = plot_roc(lr_tfidf,  x_tf_test, y_tf_test, 0, 1, 'LR TFIDF')

rf1 = plot_roc(rf_sg,     x_sg_test, y_sg_test, 1, 0, 'RF SG')
rf2 = plot_roc(rf_tfidf,  x_tf_test, y_tf_test, 1, 1, 'RF TFIDF')

svm1 = plot_roc(svm_sg,    x_sg_test, y_sg_test, 3, 0, 'SVM SG')
svm2 = plot_roc(svm_tfidf, x_tf_test, y_tf_test, 3, 1, 'SVM TFIDF')

dt1 = plot_roc(dt_sg,     x_sg_test, y_sg_test, 2, 0, 'DT SG')
dt2 = plot_roc(dt_tfidf,  x_tf_test, y_tf_test, 2, 1, 'DT TFIDF')

plt.show()

### Evaluate All the models and choose the best one

We create a dataframe that contain 5 columns
- Classifier Name
- Accuracy Score
- Precision Score
- Recall Score
- F1 Score

We evaluate which one is the best classifier by the Accuracy Score

In [ ]:
res = pd.DataFrame(columns=['Classifier Name', 'Accuracy Score', 'Train Accuracy', 'Precision Score', 'Recall Score', 'F1 Score'])

In [ ]:
#Create functions that store the evaluation of classifier in the created dataframe
def evaluate_classifier(clf, x_test, y_test, x_train, y_train, classifier_name):
    y_pred = clf.predict(x_test)
    y_train_pred = clf.predict(x_train)
    accuracy = accuracy_score(y_pred=y_pred, y_true=y_test)
    train_accuracy = accuracy_score(y_pred=y_train_pred, y_true=y_train)
    precision = precision_score(y_pred=y_pred, y_true=y_test)
    recall = recall_score(y_pred=y_pred, y_true=y_test)
    f1 = f1_score(y_pred=y_pred, y_true=y_test)
    eval_row = [classifier_name, accuracy, train_accuracy, precision, recall, f1]
    eval_row[1:] = np.round(eval_row[1:], decimals=3)
    res.loc[len(res)] = eval_row

In [ ]:
evaluate_classifier(lr_sg,    x_sg_test, y_sg_test, x_sg_train, y_sg_train, 'Logistic Regression SkipGram')
evaluate_classifier(lr_tfidf, x_tf_test, y_tf_test, x_tf_train, y_tf_train, 'Logistic Regression TF_IDF')

evaluate_classifier(rf_sg,    x_sg_test, y_sg_test, x_sg_train, y_sg_train, 'Random Forest SkipGram')
evaluate_classifier(rf_tfidf, x_tf_test, y_tf_test, x_tf_train, y_tf_train, 'Random Forest TF_IDF')

evaluate_classifier(svm_sg,    x_sg_test, y_sg_test, x_sg_train, y_sg_train, 'SVM SkipGram')
evaluate_classifier(svm_tfidf, x_tf_test, y_tf_test, x_tf_train, y_tf_train, 'SVM TF_IDF')

evaluate_classifier(dt_sg,    x_sg_test, y_sg_test, x_sg_train, y_sg_train, 'Decision Tree SkipGram')
evaluate_classifier(dt_tfidf, x_tf_test, y_tf_test, x_tf_train, y_tf_train, 'Decision Tree TF_IDF')

In [ ]:
res.sort_values(by=['Accuracy Score'], ascending=False)